# TreeSimplify Stress Notebook

This notebook focuses on **non-trivial, domain-agnostic** symbolic tests.

What this notebook does:
- compares `TreeSimplify.simplify` against `Symbolics.simplify` on harder algebraic structures
- runs deterministic and fuzz-style test batches
- shows LaTeX renderings for representative hard cases
- ends with the full end-to-end corpus check


In [1]:
import Pkg
Pkg.activate(joinpath(pwd(), ".."))
Pkg.instantiate()
println("Active project: ", Base.active_project())

  Activating project at `~/src/TreeSimplify.jl`


Active project: /home/vsilv/src/TreeSimplify.jl/Project.toml


In [2]:
using Symbolics
using Latexify
using Random
import TreeSimplify

println("Loaded TreeSimplify + Symbolics + Latexify")

Loaded TreeSimplify + Symbolics + Latexify


In [3]:
config = TreeSimplify.RunConfig(
    seed = 0x5eed,
    budget = TreeSimplify.SearchBudget(
        max_depth = 7,
        beam_width = 14,
        max_expansions = 2500,
        max_nodes = 8000,
        max_time_seconds = 4.0,
    ),
    validation = TreeSimplify.ValidationConfig(
        symbolic_first = false,
        numerical_fallback = true,
        random_samples = 10,
    ),
    novelty_penalty = 0.05,
    rule_family_throttle = 2,
    acceptance_improvement_min = 1e-9,
)

println("Config ready.")

Config ready.


In [4]:
function run_case(label, expr; show_latex=true)
    println("\n===== ", label, " =====")

    t_tree = @elapsed tree_result = TreeSimplify.simplify(expr; config=config)
    tree_expr = tree_result.best_expr

    sym_ok = true
    sym_expr = nothing
    t_sym = @elapsed begin
        try
            sym_expr = Symbolics.simplify(expr)
        catch err
            sym_ok = false
            println("Symbolics.simplify failed: ", typeof(err))
        end
    end

    raw_len = length(TreeSimplify.stable_serialize(expr))
    tree_len = length(TreeSimplify.stable_serialize(tree_expr))

    eq_tree = TreeSimplify.validate_equivalence(expr, tree_expr, config)

    println("TreeSimplify time (s): ", round(t_tree, digits=3))
    println("Symbolics time (s):    ", round(t_sym, digits=3), sym_ok ? "" : " (failed)")
    println("raw length:            ", raw_len)
    println("TreeSimplify length:   ", tree_len)
    println("score before:          ", round(tree_result.score_before, digits=4))
    println("score after:           ", round(tree_result.score_after, digits=4))
    println("accepted:              ", tree_result.accepted)
    println("validation passed:     ", tree_result.validation_passed)
    println("equivalence(raw,out):  ", eq_tree.passed, " (mode=", eq_tree.mode, ")")

    if sym_ok
        sym_len = length(TreeSimplify.stable_serialize(sym_expr))
        eq_sym = TreeSimplify.validate_equivalence(expr, sym_expr, config)
        eq_cross = TreeSimplify.validate_equivalence(tree_expr, sym_expr, config)
        println("Symbolics length:      ", sym_len)
        println("equivalence(raw,sym):  ", eq_sym.passed, " (mode=", eq_sym.mode, ")")
        println("equivalence(tree,sym): ", eq_cross.passed, " (mode=", eq_cross.mode, ")")
    end

    if show_latex
        println("\nLaTeX(raw):")
        display(latexify(expr))
        println("LaTeX(TreeSimplify output):")
        display(latexify(tree_expr))
    end

    return (
        label = label,
        raw = expr,
        tree = tree_result,
        tree_time = t_tree,
        sym_ok = sym_ok,
        sym_expr = sym_expr,
        sym_time = t_sym,
        raw_len = raw_len,
        tree_len = tree_len,
        eq_tree = eq_tree,
    )
end

run_case (generic function with 1 method)

## Hard, domain-agnostic cases

These are intentionally more complex than toy algebra drills.

In [5]:
@variables x y z a b c u v w

cases = [
    (
        "Cyclic difference-quotient sum",
        ((x^2 - y^2) / (x - y)) + ((y^2 - z^2) / (y - z)) + ((z^2 - x^2) / (z - x)),
    ),
    (
        "Nested rational 3-cycle",
        (1 + 1 / (1 + 1 / (1 + x + y))) +
        (1 + 1 / (1 + 1 / (1 + y + z))) +
        (1 + 1 / (1 + 1 / (1 + z + x))),
    ),
    (
        "High-degree mixed parity cancellation",
        (x + y + z)^7 - (x - y - z)^7 + (x + y - z)^7 - (x - y + z)^7,
    ),
    (
        "Squared cyclic difference-quotients",
        ((x^2 - y^2) / (x - y))^2 + ((y^2 - z^2) / (y - z))^2 + ((z^2 - x^2) / (z - x))^2,
    ),
    (
        "CSE-heavy repeated block",
        ((a + b + c)^2 - (a - b - c)^2) +
        ((a + b + c)^2 - (a - b - c)^2) +
        ((a + b + c)^2 - (a - b - c)^2),
    ),
    (
        "Rewrite-maze with neutral elements",
        ((((u + 0) * 1) / 1) + (((v + 0) * 1) / 1) + (((w + 0) * 1) / 1)) +
        (((u + 0) + (u + 0)) - 0) + ((v * 1) + (w * 1)),
    ),
    (
        "Nested cancellation islands",
        (((a + b) / (a + b)) + ((b + c) / (b + c)) + ((c + a) / (c + a))) * 1 + 0,
    ),
]

println("Loaded ", length(cases), " hard test cases.")

Loaded 7 hard test cases.


In [6]:
res1 = run_case(cases[1][1], cases[1][2]; show_latex=true)


===== Cyclic difference-quotient sum =====
TreeSimplify time (s): 5.05
Symbolics time (s):    51.17
raw length:            77
TreeSimplify length:   77
score before:          80.3
score after:           80.3
accepted:              false
validation passed:     true
equivalence(raw,out):  false (mode=none)
Symbolics length:      30
equivalence(raw,sym):  true (mode=numerical)
equivalence(tree,sym): true (mode=numerical)

LaTeX(raw):


L"\begin{equation}
\frac{y^{2} - z^{2}}{y - z} + \frac{x^{2} - y^{2}}{x - y} + \frac{ - x^{2} + z^{2}}{ - x + z}
\end{equation}
"

LaTeX(TreeSimplify output):


L"\begin{equation}
\frac{y^{2} - z^{2}}{y - z} + \frac{x^{2} - y^{2}}{x - y} + \frac{ - x^{2} + z^{2}}{ - x + z}
\end{equation}
"

(label = "Cyclic difference-quotient sum", raw = (y^2 - (z^2)) / (y - z) + (x^2 - (y^2)) / (x - y) + (-(x^2) + z^2) / (-x + z), tree = TreeSimplify.SimplificationResult((y^2 - (z^2)) / (y - z) + (x^2 - (y^2)) / (x - y) + (-(x^2) + z^2) / (-x + z), (y^2 - (z^2)) / (y - z) + (x^2 - (y^2)) / (x - y) + (-(x^2) + z^2) / (-x + z), false, 80.3, 80.3, true, TreeSimplify.SearchStats(0, 1, 1, :time_budget), TreeSimplify.TraceBuffer(TreeSimplify.TraceEvent[TreeSimplify.TraceEvent(:run_started, (seed = 0x0000000000005eed, beam_width = 14)), TreeSimplify.TraceEvent(:validation, (passed = true, mode = :numerical, worst_abs = 0.0, worst_rel = 0.0)), TreeSimplify.TraceEvent(:run_finished, (reason = :time_budget, accepted = false, score_before = 80.3, score_after = 80.3))])), tree_time = 5.050038765, sym_ok = true, sym_expr = (2//1)*x + (2//1)*y + (2//1)*z, sym_time = 51.170016563, raw_len = 77, tree_len = 77, eq_tree = TreeSimplify.ValidationReport(false, :none, Inf, Inf, 0))

In [7]:
res2 = run_case(cases[2][1], cases[2][2]; show_latex=true)


===== Nested rational 3-cycle =====
TreeSimplify time (s): 2.752
Symbolics time (s):    1.285
raw length:            85
TreeSimplify length:   85
score before:          91.1
score after:           91.1
accepted:              false
validation passed:     true
equivalence(raw,out):  false (mode=none)
Symbolics length:      192
equivalence(raw,sym):  true (mode=numerical)
equivalence(tree,sym): true (mode=numerical)

LaTeX(raw):


L"\begin{equation}
3 + \frac{1}{1 + \frac{1}{1 + x + z}} + \frac{1}{1 + \frac{1}{1 + x + y}} + \frac{1}{1 + \frac{1}{1 + y + z}}
\end{equation}
"

LaTeX(TreeSimplify output):


L"\begin{equation}
3 + \frac{1}{1 + \frac{1}{1 + x + z}} + \frac{1}{1 + \frac{1}{1 + x + y}} + \frac{1}{1 + \frac{1}{1 + y + z}}
\end{equation}
"

(label = "Nested rational 3-cycle", raw = 3 + 1 / (1 + 1 / (1 + x + z)) + 1 / (1 + 1 / (1 + x + y)) + 1 / (1 + 1 / (1 + y + z)), tree = TreeSimplify.SimplificationResult(3 + 1 / (1 + 1 / (1 + x + z)) + 1 / (1 + 1 / (1 + x + y)) + 1 / (1 + 1 / (1 + y + z)), 3 + 1 / (1 + 1 / (1 + x + z)) + 1 / (1 + 1 / (1 + x + y)) + 1 / (1 + 1 / (1 + y + z)), false, 91.1, 91.1, true, TreeSimplify.SearchStats(0, 1, 2, :frontier_exhausted), TreeSimplify.TraceBuffer(TreeSimplify.TraceEvent[TreeSimplify.TraceEvent(:run_started, (seed = 0x0000000000005eed, beam_width = 14)), TreeSimplify.TraceEvent(:depth_completed, (depth = 1, frontier = 0, visited = 1)), TreeSimplify.TraceEvent(:validation, (passed = true, mode = :numerical, worst_abs = 0.0, worst_rel = 0.0)), TreeSimplify.TraceEvent(:run_finished, (reason = :frontier_exhausted, accepted = false, score_before = 91.1, score_after = 91.1))])), tree_time = 2.752266644, sym_ok = true, sym_expr = (36 + 40x + 40y + 40z + 11(x^2) + 33x*y + 33x*z + 11(y^2) + 33y*z

In [8]:
res3 = run_case(cases[3][1], cases[3][2]; show_latex=false)


===== High-degree mixed parity cancellation =====
TreeSimplify time (s): 1.251
Symbolics time (s):    0.524
raw length:            66
TreeSimplify length:   66
score before:          49.6
score after:           49.6
accepted:              false
validation passed:     true
equivalence(raw,out):  false (mode=none)
Symbolics length:      66
equivalence(raw,sym):  false (mode=none)
equivalence(tree,sym): false (mode=none)


(label = "High-degree mixed parity cancellation", raw = -((x - y + z)^7) - ((x - y - z)^7) + (x + y - z)^7 + (x + y + z)^7, tree = TreeSimplify.SimplificationResult(-((x - y + z)^7) - ((x - y - z)^7) + (x + y - z)^7 + (x + y + z)^7, -((x - y + z)^7) - ((x - y - z)^7) + (x + y - z)^7 + (x + y + z)^7, false, 49.6, 49.6, true, TreeSimplify.SearchStats(0, 1, 2, :frontier_exhausted), TreeSimplify.TraceBuffer(TreeSimplify.TraceEvent[TreeSimplify.TraceEvent(:run_started, (seed = 0x0000000000005eed, beam_width = 14)), TreeSimplify.TraceEvent(:depth_completed, (depth = 1, frontier = 0, visited = 1)), TreeSimplify.TraceEvent(:validation, (passed = true, mode = :numerical, worst_abs = 0.0, worst_rel = 0.0)), TreeSimplify.TraceEvent(:run_finished, (reason = :frontier_exhausted, accepted = false, score_before = 49.6, score_after = 49.6))])), tree_time = 1.251367232, sym_ok = true, sym_expr = -((x - y + z)^7) - ((x - y - z)^7) + (x + y - z)^7 + (x + y + z)^7, sym_time = 0.524186722, raw_len = 66, tr

In [9]:
res4 = run_case(cases[4][1], cases[4][2]; show_latex=false)


===== Squared cyclic difference-quotients =====
TreeSimplify time (s): 0.014
Symbolics time (s):    0.755
raw length:            89
TreeSimplify length:   89
score before:          89.0
score after:           89.0
accepted:              false
validation passed:     true
equivalence(raw,out):  false (mode=none)
Symbolics length:      81
equivalence(raw,sym):  true (mode=numerical)
equivalence(tree,sym): true (mode=numerical)


(label = "Squared cyclic difference-quotients", raw = ((y^2 - (z^2)) / (y - z))^2 + ((x^2 - (y^2)) / (x - y))^2 + ((-(x^2) + z^2) / (-x + z))^2, tree = TreeSimplify.SimplificationResult(((y^2 - (z^2)) / (y - z))^2 + ((x^2 - (y^2)) / (x - y))^2 + ((-(x^2) + z^2) / (-x + z))^2, ((y^2 - (z^2)) / (y - z))^2 + ((x^2 - (y^2)) / (x - y))^2 + ((-(x^2) + z^2) / (-x + z))^2, false, 89.0, 89.0, true, TreeSimplify.SearchStats(0, 1, 2, :frontier_exhausted), TreeSimplify.TraceBuffer(TreeSimplify.TraceEvent[TreeSimplify.TraceEvent(:run_started, (seed = 0x0000000000005eed, beam_width = 14)), TreeSimplify.TraceEvent(:depth_completed, (depth = 1, frontier = 0, visited = 1)), TreeSimplify.TraceEvent(:validation, (passed = true, mode = :numerical, worst_abs = 0.0, worst_rel = 0.0)), TreeSimplify.TraceEvent(:run_finished, (reason = :frontier_exhausted, accepted = false, score_before = 89.0, score_after = 89.0))])), tree_time = 0.013692557, sym_ok = true, sym_expr = (2//1)*(x^2) + (2//1)*x*y + (2//1)*x*z + 

In [10]:
res5 = run_case(cases[5][1], cases[5][2]; show_latex=false)


===== CSE-heavy repeated block =====
TreeSimplify time (s): 0.004
Symbolics time (s):    0.004
raw length:            36
TreeSimplify length:   36
score before:          26.0
score after:           26.0
accepted:              false
validation passed:     true
equivalence(raw,out):  false (mode=none)
Symbolics length:      36
equivalence(raw,sym):  false (mode=none)
equivalence(tree,sym): false (mode=none)


(label = "CSE-heavy repeated block", raw = -3((a - b - c)^2) + 3((a + b + c)^2), tree = TreeSimplify.SimplificationResult(-3((a - b - c)^2) + 3((a + b + c)^2), -3((a - b - c)^2) + 3((a + b + c)^2), false, 26.0, 26.0, true, TreeSimplify.SearchStats(0, 1, 2, :frontier_exhausted), TreeSimplify.TraceBuffer(TreeSimplify.TraceEvent[TreeSimplify.TraceEvent(:run_started, (seed = 0x0000000000005eed, beam_width = 14)), TreeSimplify.TraceEvent(:depth_completed, (depth = 1, frontier = 0, visited = 1)), TreeSimplify.TraceEvent(:validation, (passed = true, mode = :numerical, worst_abs = 0.0, worst_rel = 0.0)), TreeSimplify.TraceEvent(:run_finished, (reason = :frontier_exhausted, accepted = false, score_before = 26.0, score_after = 26.0))])), tree_time = 0.004487595, sym_ok = true, sym_expr = -3((a - b - c)^2) + 3((a + b + c)^2), sym_time = 0.003951453, raw_len = 36, tree_len = 36, eq_tree = TreeSimplify.ValidationReport(false, :none, Inf, Inf, 0))

In [11]:
res6 = run_case(cases[6][1], cases[6][2]; show_latex=true)


===== Rewrite-maze with neutral elements =====
TreeSimplify time (s): 0.003
Symbolics time (s):    0.017
raw length:            12
TreeSimplify length:   12
score before:          11.9
score after:           11.9
accepted:              false
validation passed:     true
equivalence(raw,out):  false (mode=none)
Symbolics length:      13
equivalence(raw,sym):  false (mode=numerical)
equivalence(tree,sym): false (mode=numerical)

LaTeX(raw):


L"\begin{equation}
3 u + 2 v + 2 w
\end{equation}
"

LaTeX(TreeSimplify output):


L"\begin{equation}
3 u + 2 v + 2 w
\end{equation}
"

(label = "Rewrite-maze with neutral elements", raw = 3u + 2v + 2w, tree = TreeSimplify.SimplificationResult(3u + 2v + 2w, 3u + 2v + 2w, false, 11.9, 11.9, true, TreeSimplify.SearchStats(0, 1, 2, :frontier_exhausted), TreeSimplify.TraceBuffer(TreeSimplify.TraceEvent[TreeSimplify.TraceEvent(:run_started, (seed = 0x0000000000005eed, beam_width = 14)), TreeSimplify.TraceEvent(:depth_completed, (depth = 1, frontier = 0, visited = 1)), TreeSimplify.TraceEvent(:validation, (passed = true, mode = :numerical, worst_abs = 0.0, worst_rel = 0.0)), TreeSimplify.TraceEvent(:run_finished, (reason = :frontier_exhausted, accepted = false, score_before = 11.9, score_after = 11.9))])), tree_time = 0.003111611, sym_ok = true, sym_expr = 3u + 2(v + w), sym_time = 0.017002148, raw_len = 12, tree_len = 12, eq_tree = TreeSimplify.ValidationReport(false, :none, Inf, Inf, 0))

In [12]:
res7 = run_case(cases[7][1], cases[7][2]; show_latex=true)


===== Nested cancellation islands =====
TreeSimplify time (s): 0.304
Symbolics time (s):    0.02
raw length:            1
TreeSimplify length:   1
score before:          1.0
score after:           1.0
accepted:              false
validation passed:     true
equivalence(raw,out):  false (mode=none)
Symbolics length:      1
equivalence(raw,sym):  false (mode=none)
equivalence(tree,sym): false (mode=none)

LaTeX(raw):


L"\begin{equation}
3
\end{equation}
"

LaTeX(TreeSimplify output):


L"$3$"

(label = "Nested cancellation islands", raw = 3, tree = TreeSimplify.SimplificationResult(3, 3, false, 1.0, 1.0, true, TreeSimplify.SearchStats(0, 1, 2, :frontier_exhausted), TreeSimplify.TraceBuffer(TreeSimplify.TraceEvent[TreeSimplify.TraceEvent(:run_started, (seed = 0x0000000000005eed, beam_width = 14)), TreeSimplify.TraceEvent(:depth_completed, (depth = 1, frontier = 0, visited = 1)), TreeSimplify.TraceEvent(:validation, (passed = true, mode = :numerical, worst_abs = 0.0, worst_rel = 0.0)), TreeSimplify.TraceEvent(:run_finished, (reason = :frontier_exhausted, accepted = false, score_before = 1.0, score_after = 1.0))])), tree_time = 0.304043908, sym_ok = true, sym_expr = 3, sym_time = 0.019994491, raw_len = 1, tree_len = 1, eq_tree = TreeSimplify.ValidationReport(false, :none, Inf, Inf, 0))

## Batch summary table

In [13]:
all_results = [res1, res2, res3, res4, res5, res6, res7]

println("label"^70)
println("raw_len  out_len  tree_t(s)  sym_t(s)  accepted  valid")
println("-"^80)
for r in all_results
    println(rpad(r.label, 44),
        lpad(r.raw_len, 7), "  ",
        lpad(r.tree_len, 7), "  ",
        lpad(round(r.tree_time, digits=3), 8), "  ",
        lpad(round(r.sym_time, digits=3), 8), "  ",
        lpad(r.tree.accepted, 7), "  ",
        lpad(r.eq_tree.passed, 5))
end

println()
println("validated count: ", count(r -> r.eq_tree.passed, all_results), "/", length(all_results))
println("accepted count:  ", count(r -> r.tree.accepted, all_results), "/", length(all_results))
println("Tree faster count:", count(r -> r.tree_time < r.sym_time, all_results), "/", length(all_results))

labellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabellabel
raw_len  out_len  tree_t(s)  sym_t(s)  accepted  valid
--------------------------------------------------------------------------------
Cyclic difference-quotient sum                   77       77      5.05     51.17    false  false
Nested rational 3-cycle                          85       85     2.752     1.285    false  false
High-degree mixed parity cancellation            66       66     1.251     0.524    false  false
Squared cyclic difference-quotients              89       89     0.014     0.755    false  false
CSE-heavy repeated block                         36       36     0.004     0.004    false  false
Rewrite-maze with neutral el

## Determinism test battery

In [14]:
det_ok = true
for (lbl, expr) in cases
    rA = TreeSimplify.simplify(expr; config=config)
    rB = TreeSimplify.simplify(expr; config=config)
    hA = TreeSimplify.structural_hash(rA.best_expr)
    hB = TreeSimplify.structural_hash(rB.best_expr)
    same = (hA == hB) && (rA.score_after == rB.score_after)
    println(rpad(lbl, 44), " deterministic=", same)
    det_ok &= same
end
println("\nDeterminism batch passed: ", det_ok)

Cyclic difference-quotient sum               deterministic=true
Nested rational 3-cycle                      deterministic=true
High-degree mixed parity cancellation        deterministic=true
Squared cyclic difference-quotients          deterministic=true
CSE-heavy repeated block                     deterministic=true
Rewrite-maze with neutral elements           deterministic=true
Nested cancellation islands                  deterministic=true

Determinism batch passed: true


## Fuzz-style test sweep (many random expressions)

In [15]:
rng = MersenneTwister(0x1234)
vars = [x, y, z]

function rand_leaf(rng, vars)
    if rand(rng) < 0.6
        return vars[rand(rng, 1:length(vars))]
    else
        return rand(rng, -3:3)
    end
end

function rand_expr(rng, vars, depth)
    if depth <= 0
        return rand_leaf(rng, vars)
    end
    op = rand(rng, [:+, :-, :*, :/])
    a = rand_expr(rng, vars, depth - 1)
    b = rand_expr(rng, vars, depth - 1)
    b = (b == 0 ? 1 : b)
    expr = op === :+ ? (a + b) :
           op === :- ? (a - b) :
           op === :* ? (a * b) : (a / b)
    if rand(rng) < 0.4
        expr = (expr + 0) * 1
    end
    if rand(rng) < 0.25
        expr = expr / 1
    end
    return expr
end

N = 40
valid_count = 0
accepted_count = 0
improved_count = 0

for i in 1:N
    expr = rand_expr(rng, vars, 3)
    r = TreeSimplify.simplify(expr; config=config)
    rep = TreeSimplify.validate_equivalence(expr, r.best_expr, config)
    valid_count += rep.passed ? 1 : 0
    accepted_count += r.accepted ? 1 : 0
    improved_count += (r.score_after < r.score_before) ? 1 : 0
end

println("Fuzz tests run:      ", N)
println("Validation passed:   ", valid_count, "/", N)
println("Accepted:            ", accepted_count, "/", N)
println("Score improved:      ", improved_count, "/", N)

LoadError: TypeError: non-boolean (Num) used in boolean context
A symbolic expression appeared in a Boolean context. This error arises in situations where Julia expects a Bool, like 
[34mif boolean_condition[39m[32m		 use ifelse(boolean_condition, then branch, else branch)[39m
[34mx && y[39m[32m				 use x & y[39m
[34mboolean_condition ? a : b[39m[32m	 use ifelse(boolean_condition, a, b)[39m
but a symbolic expression appeared instead of a Bool. For help regarding control flow with symbolic variables, see https://docs.sciml.ai/ModelingToolkit/dev/basics/FAQ/#How-do-I-handle-if-statements-in-my-symbolic-forms?

  Julia 1.12 has introduced more strict world age semantics for global bindings.
  !!! This code may malfunction under Revise.
  !!! This code will error in future versions of Julia.
Hint: Add an appropriate `invokelatest` around the access to this binding.
To make this warning an error, and hence obtain a stack trace, use `julia --depwarn=error`.


## Final test: full corpus end-to-end

In [16]:
println("Running end-to-end validation on expression corpus...")
e2e_config = TreeSimplify.RunConfig(
    budget = TreeSimplify.SearchBudget(max_depth = 4, beam_width = 8, max_expansions = 600, max_nodes = 2000, max_time_seconds = 30.0),
    validation = TreeSimplify.ValidationConfig(symbolic_first = false, numerical_fallback = true, random_samples = 6),
    acceptance_improvement_min = 1e-9,
)

e2e = TreeSimplify.run_end_to_end_validation(config = e2e_config)

println("total sections:          ", e2e.total)
println("input equivalence passed:", e2e.input_equivalence_passed)
println("output equivalence passed:", e2e.output_equivalence_passed)
println()
for r in e2e.records
    println(r.label,
        " | in_eq=", r.input_equivalent_to_expected,
        " | out_eq=", r.output_equivalent_to_expected,
        " | accepted=", r.accepted,
        " | score: ", round(r.score_before, digits=2), " -> ", round(r.score_after, digits=2))
end

Running end-to-end validation on expression corpus...
total sections:          9
input equivalence passed:9
output equivalence passed:9

a² | in_eq=true | out_eq=true | accepted=true | score: 6591.8 -> 6588.35
a†a | in_eq=true | out_eq=true | accepted=true | score: 11626.6 -> 11619.4
a†a³ | in_eq=true | out_eq=true | accepted=true | score: 5292.6 -> 5282.3
a†² | in_eq=true | out_eq=true | accepted=true | score: 6659.9 -> 6653.15
a†²a² | in_eq=true | out_eq=true | accepted=true | score: 7710.5 -> 7694.3
a†³a | in_eq=true | out_eq=true | accepted=true | score: 5510.0 -> 5501.5
a†⁴ | in_eq=true | out_eq=true | accepted=true | score: 1504.4 -> 1502.5
a⁴ | in_eq=true | out_eq=true | accepted=true | score: 1411.1 -> 1407.8
identity | in_eq=true | out_eq=true | accepted=true | score: 4123.1 -> 4118.15
